# AI651 PA2 — Question 2: Motion or Memory?
ConvLSTM forecasting on Moving MNIST, symmetry and speed stress tests, models A–D, and the symmetrised final model.

## Running on Kaggle
1. **Settings → Accelerator: GPU (T4 or P100)** and **Internet: ON** (the dataset downloads from cs.toronto.edu).
   If the download fails, attach any Kaggle dataset that contains `mnist_test_seq.npy`; it is picked up automatically.
2. The handout requires that **Q2.5 and Q2.11 are committed to GitHub before** running the matching experiments. The notebook enforces this with two gates in the config cell:

| Stage | Runs | Unlocked by |
|---|---|---|
| 1 | Table 1, Fig 1, train model A (2 seeds), Table 2, Fig 2 | always |
| 2 | Testing the Baseline: Tables 3–4, Figs 3–5 (+ GIFs) | `Q2_5_COMMITTED = True` |
| 3 | Models B, C, D (+D+B), Tables 5–6, Fig 6 (+ GIFs) | `Q2_11_COMMITTED = True` |

Locked cells (marked `%%stage N`) print *skipped* instead of failing, so **Save Version → Save & Run All** always completes.

**Typical workflow**
- *Run 1:* both gates `False`. Save & Run All (~30 min on a T4). Write Q2.3/Q2.4, then **write Q2.5 and commit it**.
- *Run 2:* set `Q2_5_COMMITTED = True`. Save & Run All. Write Q2.6–Q2.10, then **write Q2.11 and commit it**.
- *Run 3:* set `Q2_11_COMMITTED = True`. Save & Run All (~2.5–3 h on a T4).

**Reusing checkpoints across runs:** after a run, use *Add Input → Notebook Output* and attach the previous version of this notebook. Any `pa2_q2/ckpt` or `pa2_q2/results` files found under `/kaggle/input` are copied in and not retrained.
In an interactive session you can instead flip a gate and keep running cells: trained models stay in memory and on disk.

Everything is written to `/kaggle/working/pa2_q2/` (`figs/`, `gifs/`, `tables/`, `ckpt/`), and `pa2_q2.zip` is created at the end.

In [ ]:
import os
# ============================ configuration ============================
# --- commit gates (see the table above) ---
Q2_5_COMMITTED  = False   # set True only after Q2.5 is committed to your repo
Q2_11_COMMITTED = False   # set True only after Q2.11 is committed to your repo

# --- Section 4 / 5 choices ---
RUN_D_SELECTION = True    # compare model-D candidates on the VALIDATION split (seed 0, short runs)
SEL_EPOCHS      = 8       # epochs per candidate during selection
D_DEFAULT       = 'D_k5'  # used if RUN_D_SELECTION = False
TRAIN_DB        = True    # also train the optional D+B model
FINAL           = 'D'     # 'D' or 'D+B': the model wrapped as the final symmetrised forecaster

# --- protocol from the handout (do not change for the real run) ---
EPOCHS, BATCH, LR, SEEDS = 30, 32, 1e-3, [0, 1]
N_TEST = 1000
MAX_BATCHES = None

# --- local smoke-test switches (ignored on Kaggle unless the env vars are set) ---
QUICK = os.environ.get('PA2_QUICK', '0') == '1'
if QUICK:
    EPOCHS, SEL_EPOCHS, MAX_BATCHES, N_TEST = 1, 1, 3, 64
if os.environ.get('PA2_UNLOCK', '0') == '1':
    Q2_5_COMMITTED = Q2_11_COMMITTED = True

ON_KAGGLE = os.path.isdir('/kaggle/working')
OUT = '/kaggle/working/pa2_q2' if ON_KAGGLE else os.path.abspath('pa2_q2_out' + ('_quick' if QUICK else ''))
DATA_ROOT = os.environ.get('PA2_DATA_ROOT', '/kaggle/working/data' if ON_KAGGLE else os.path.abspath('data'))

In [ ]:
import sys, json, time, glob, shutil, random, platform
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
import matplotlib.pyplot as plt
import cv2
from PIL import Image
from IPython.display import display, Markdown
from IPython.core.magic import register_cell_magic

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.backends.cudnn.benchmark = True
for d in ('ckpt', 'figs', 'gifs', 'tables', 'results'):
    os.makedirs(f'{OUT}/{d}', exist_ok=True)

HARDWARE = (torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else (platform.processor() or platform.machine()) + ' CPU')
print(f'torch {torch.__version__} | device {DEVICE} | hardware: {HARDWARE} | QUICK={QUICK}')
print('output dir:', OUT)

# restore checkpoints / cached results from an attached previous notebook version
if ON_KAGGLE:
    for sub in ('ckpt', 'results'):
        for f in glob.glob(f'/kaggle/input/**/pa2_q2/{sub}/*', recursive=True):
            dst = f'{OUT}/{sub}/{os.path.basename(f)}'
            if not os.path.exists(dst):
                shutil.copy(f, dst); print('restored', dst)

def stage_unlocked(n):
    return n <= 1 or (n == 2 and Q2_5_COMMITTED) or (n >= 3 and Q2_5_COMMITTED and Q2_11_COMMITTED)

@register_cell_magic
def stage(line, cell):
    n = int(line.strip())
    if not stage_unlocked(n):
        print(f'⏸  Stage {n} is locked (see the config cell). Cell skipped.')
        return
    r = get_ipython().run_cell(cell)
    if not r.success:
        raise RuntimeError(f'stage {n} cell failed (see the traceback above)')

def sync():
    if DEVICE.type == 'cuda': torch.cuda.synchronize()

def fmt(v):
    """3 significant figures; scientific notation below 1e-3."""
    v = float(v)
    if v == 0: return '0'
    if abs(v) < 1e-3: return f'{v:.2e}'
    s = f'{v:#.3g}'                          # keep trailing zeros (0.240, not 0.24)
    return s[:-1] if s.endswith('.') else s

def ftime(v): return f'{float(v):.2f}'

def fmt_sci(v):
    v = float(v)
    return '0' if v == 0 else f'{v:.2e}'

def cell(vals, f=fmt):
    """'mean (seed 0, seed 1)' for learned models, a single value for deterministic ones."""
    vals = [float(v) for v in vals]
    if len(vals) == 1: return f(vals[0])
    return f'{f(np.mean(vals))} ({", ".join(f(v) for v in vals)})'

def save_table(df, name):
    df.to_csv(f'{OUT}/tables/{name}.csv', index=False)
    try: txt = df.to_markdown(index=False)
    except Exception: txt = df.to_string(index=False)
    with open(f'{OUT}/tables/{name}.md', 'w') as fh: fh.write(txt + '\n')
    display(Markdown(f'**{name.replace("_", " ")}**')); display(df)

---
# 1. Setup
### Data
Split by video index: 0–7,999 train, 8,000–8,999 validation, 9,000–9,999 test. Videos are kept as `uint8` on the device and converted to `float32 / 255` per batch.

In [ ]:
from torchvision.datasets import MovingMNIST

os.makedirs(f'{DATA_ROOT}/MovingMNIST', exist_ok=True)
_npy = f'{DATA_ROOT}/MovingMNIST/mnist_test_seq.npy'
if not os.path.exists(_npy):
    hits = glob.glob('/kaggle/input/**/mnist_test_seq.npy', recursive=True)
    if hits: shutil.copy(hits[0], _npy); print('copied dataset from', hits[0])

ds = MovingMNIST(root=DATA_ROOT, split=None, download=True)
VID = ds.data                                    # (10000, 20, 1, 64, 64) uint8
assert tuple(VID.shape) == (10000, 20, 1, 64, 64) and VID.dtype == torch.uint8

train_u8 = VID[:8000].to(DEVICE)
val_u8   = VID[8000:9000].to(DEVICE)
test_u8  = VID[9000:9000 + N_TEST].to(DEVICE)    # test video i  <->  dataset index 9000 + i

def to_f(u8):
    return u8.float().div(255.)

print('train', tuple(train_u8.shape), '| val', tuple(val_u8.shape), '| test', tuple(test_u8.shape))

### Transformations and windows
All experiments call these functions. Frames are numbered 1–20, as in the handout.
`G = {R^a, M R^a : a = 0..3}`, with `M R^a(x) = M(R^a(x))` (rotate first, then mirror).

In [ ]:
def M(x):              # mirror: (Mx)[i, j] = x[i, 63 - j]
    return x.flip(-1)

def R(x, a=1):         # rotate a * 90 degrees anticlockwise
    a %= 4
    return torch.rot90(x, k=a, dims=(-2, -1)) if a else x

G = [(m, a) for m in (0, 1) for a in range(4)]          # (0, a) = R^a, (1, a) = M R^a
G_NAMES = [('M' if m else '') + f'R{a}' for m, a in G]

def apply_g(x, g):
    m, a = g
    y = R(x, a)
    return M(y) if m else y

def inv_g(g):
    m, a = g
    return g if m else (0, (-a) % 4)                     # M R^a is its own inverse

def make_windows(vids, t, k):
    """vids: (B, 20, 1, 64, 64); t, k: int or (B,) tensors (frame numbers are 1-based).
    Returns X (B,5,1,64,64), Y (B,5,1,64,64), mask (B,5) of target frames that exist, frame numbers (B,10).
    Missing target frames are filled with a copy of frame 20; they are masked out of every loss and metric."""
    B, dev = vids.shape[0], vids.device
    t = torch.as_tensor(t, device=dev).long().reshape(-1).expand(B)
    k = torch.as_tensor(k, device=dev).long().reshape(-1).expand(B)
    fr = t[:, None] + torch.arange(10, device=dev)[None, :] * k[:, None]
    valid = fr <= 20
    sel = vids[torch.arange(B, device=dev)[:, None], fr.clamp(max=20) - 1]
    if sel.dtype == torch.uint8: sel = to_f(sel)
    return sel[:, :5], sel[:, 5:], valid[:, 5:].float(), fr

def allowed_starts(k, speed_aug):
    if not speed_aug:
        assert k == 1; return list(range(1, 12))           # all five targets exist
    return list(range(1, 20 - 6 * k + 1))                  # five inputs + at least two targets

for k in (1, 2, 3):
    print(f'{k}x speed-aug starts: {allowed_starts(k, True)}')

### Table 1: implementation checks

In [ ]:
torch.manual_seed(0); x = torch.rand(20, 1, 64, 64)
p = torch.zeros(20, 1, 64, 64); p[7, 0, 10, 20] = 1

def exact(a, b):
    return f'{(a - b).abs().max().item():g} (torch.equal={torch.equal(a, b)})'

def bright(v):
    f, _, i, j = (v == 1).nonzero()[0].tolist(); return f'({f},{i},{j})'

R4 = R(R(R(R(x))))                                   # four separate 90-degree rotations
ramp = torch.arange(1, 21, dtype=torch.float32).view(20, 1, 1, 1).expand(20, 1, 64, 64)

def frames_used(k):
    X, Y, mask, fr = make_windows(ramp[None], 1, k)
    xi = [int(v) for v in X[0, :, 0, 0, 0]]
    yi = [int(v) for v, ok in zip(Y[0, :, 0, 0, 0], mask[0]) if ok]
    return ', '.join(map(str, xi)) + ' / ' + ', '.join(map(str, yi))

table1 = pd.DataFrame([
    ['(a) max|M(M(x)) - x|', '0', exact(M(M(x)), x)],
    ['(b) max|R^4(x) - x|', '0', exact(R4, x)],
    ['(c) max|M R M(x) - R^-1(x)|', '0', exact(M(R(M(x))), R(x, -1))],
    ['(d) bright pixel of M(p) (frame,row,col)', '(7,10,43)', bright(M(p))],
    ['(e) bright pixel of R(p) (frame,row,col)', '(7,43,10)', bright(R(p))],
    ['(f) 1x input / target frames', '1, 2, 3, 4, 5 / 6, 7, 8, 9, 10', frames_used(1)],
    ['(g) 2x input / target frames', '1, 3, 5, 7, 9 / 11, 13, 15, 17, 19', frames_used(2)],
    ['(h) 3x input / target frames', '1, 4, 7, 10, 13 / 16, 19', frames_used(3)],
], columns=['Check', 'Expected', 'Yours'])
save_table(table1, 'Table_1')

### Figure 1: optical flow on the first 50 training videos
Farneback with the exact handout settings, on `uint8` frames. A pixel is kept if its intensity is > 0.2 in both frames and its flow magnitude is > 0.5 px.
The extra printout compares the direction histogram with its mirrored and 90°-rotated versions (useful for Q2.1).

In [ ]:
dxs, dys, n_pairs = [], [], 0
for v in range(50):                                      # training videos 0..49
    fr = to_f(VID[v, :, 0]).numpy()                      # [0, 1] floats
    u8 = (fr * 255).round().astype(np.uint8)
    for s in range(19):
        flow = cv2.calcOpticalFlowFarneback(u8[s], u8[s + 1], None, 0.5, 3, 15, 3, 5, 1.2, 0)
        dx, dy = flow[..., 0], flow[..., 1]
        keep = (fr[s] > 0.2) & (fr[s + 1] > 0.2) & (np.hypot(dx, dy) > 0.5)
        dxs.append(dx[keep]); dys.append(dy[keep]); n_pairs += 1
dx, dy = np.concatenate(dxs), np.concatenate(dys)
theta = np.degrees(np.arctan2(-dy, dx)) % 360
speed = np.hypot(dx, dy)
N_KEPT, MED_SPEED = len(speed), float(np.median(speed))
assert n_pairs == 950

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
w = np.full(N_KEPT, 1 / N_KEPT)
ax[0].hist(theta, bins=np.arange(0, 361, 20), weights=w, edgecolor='k')
ax[0].axhline(1 / 18, color='r', ls='--', label='uniform (1/18)')
ax[0].set_xticks(range(0, 361, 40)); ax[0].set_xlabel('flow direction θ (deg; 0° = right, 90° = up)')
ax[0].set_ylabel('fraction of kept pixels'); ax[0].set_title('(a) direction'); ax[0].legend()
ax[1].hist(speed, bins=60, weights=w, edgecolor='k')
ax[1].axvline(MED_SPEED, color='r', ls='--', label=f'median = {MED_SPEED:.3g} px/frame')
ax[1].set_xlabel('flow speed (px/frame)'); ax[1].set_ylabel('fraction of kept pixels'); ax[1].set_title('(b) speed'); ax[1].legend()
fig.suptitle(f'Figure 1: Farneback flow, first 50 training videos (950 frame pairs); '
             f'{N_KEPT:,} pixels kept, median speed {MED_SPEED:.3g} px/frame')
fig.tight_layout(); fig.savefig(f'{OUT}/figs/fig1_flow.png', dpi=150); plt.show()

# ---- diagnostics for Q2.1 ----
def hist18(th): return np.histogram(th % 360, bins=np.arange(0, 361, 20))[0] / len(th)
h0 = hist18(theta)
tv = lambda a, b: 0.5 * np.abs(a - b).sum()
print(f'kept pixels: {N_KEPT:,} | median speed {MED_SPEED:.3g} px/frame | '
      f'p10/p90 {np.percentile(speed, 10):.3g}/{np.percentile(speed, 90):.3g}')
print(f'2x ≈ {2 * MED_SPEED:.3g} px/frame, 3x ≈ {3 * MED_SPEED:.3g} px/frame (median)')
print(f'max bin {h0.max():.3f}, min bin {h0.min():.3f} (uniform = {1 / 18:.3f})')
print(f'total-variation distance to the mirrored histogram  (θ -> 180-θ): {tv(h0, hist18(180 - theta)):.3f}')
print(f'total-variation distance to the rotated  histogram  (θ -> θ+90): {tv(h0, hist18(theta + 90)):.3f}')
print(f'total-variation distance to uniform: {tv(h0, np.full(18, 1 / 18)):.3f}')

---
# 2. Baseline: ConvLSTM (model A)
`pixel_unshuffle(4)` → two ConvLSTM layers (64 channels, 3×3, no peepholes) → 1×1 conv 64→16 → `pixel_shuffle(4)` → sigmoid.
The rollout runs 9 steps with no teacher forcing: from step 6 on, the model's own prediction is fed back, and the outputs of steps 5–9 are returned.
The same class, with a different patch size, kernel size or dilation, is used for model D.

In [ ]:
class ConvLSTMCell(nn.Module):
    def __init__(self, cin, hid, k=3, dil=1):
        super().__init__()
        self.hid = hid
        self.conv = nn.Conv2d(cin + hid, 4 * hid, k, stride=1, padding=dil * (k - 1) // 2, dilation=dil, bias=True)

    def forward(self, u, h, c):
        i, f, o, g = self.conv(torch.cat([u, h], 1)).chunk(4, 1)        # one conv -> all four gates
        c = torch.sigmoid(f) * c + torch.sigmoid(i) * torch.tanh(g)
        h = torch.sigmoid(o) * torch.tanh(c)
        return h, c

class ConvLSTMForecaster(nn.Module):
    def __init__(self, patch=4, hidden=(64, 64), kernels=(3, 3), dilations=(1, 1)):
        super().__init__()
        self.patch, cin = patch, patch * patch
        cells, c = [], cin
        for h, k, d in zip(hidden, kernels, dilations):
            cells.append(ConvLSTMCell(c, h, k, d)); c = h
        self.cells = nn.ModuleList(cells)
        self.out = nn.Conv2d(c, cin, 1, bias=True)

    def init_states(self, B, dev, dtype):
        H = 64 // self.patch
        return [(torch.zeros(B, cl.hid, H, H, device=dev, dtype=dtype),) * 2 for cl in self.cells]

    def step_folded(self, u, states):            # folded input -> folded output (before shuffle / sigmoid)
        new = []
        for cl, (h, c) in zip(self.cells, states):
            h, c = cl(u, h, c); new.append((h, c)); u = h
        return self.out(u), new

    def step(self, frame, states):
        y, states = self.step_folded(F.pixel_unshuffle(frame, self.patch), states)
        return torch.sigmoid(F.pixel_shuffle(y, self.patch)), states

    def forward(self, X):                        # X: (B, 5, 1, 64, 64) -> Ŷ: (B, 5, 1, 64, 64)
        states = self.init_states(X.shape[0], X.device, X.dtype)
        preds, y = [], None
        for tau in range(1, 10):
            u = X[:, tau - 1] if tau <= 5 else y   # own prediction fed back from step 6 on (gradients flow)
            y, states = self.step(u, states)
            if tau >= 5: preds.append(y)
        return torch.stack(preds, 1)

def n_params(m):
    return sum(p.numel() for p in m.parameters() if p.requires_grad)

def measure_reach(arch):
    """Empirical one-step reach: gradient of the centre output cell w.r.t. the new input and every previous state.
    Distances are centre-to-centre on the folded grid x patch size (patch width not added), per axis (Chebyshev)."""
    torch.manual_seed(0)
    m = ConvLSTMForecaster(**arch).double()
    p = m.patch; H = 64 // p; ctr = H // 2
    u = torch.randn(1, p * p, H, H, dtype=torch.double, requires_grad=True)
    srcs, states = {'u_τ': u}, []
    for l, cl in enumerate(m.cells, 1):
        h = torch.randn(1, cl.hid, H, H, dtype=torch.double, requires_grad=True)
        c = torch.randn(1, cl.hid, H, H, dtype=torch.double, requires_grad=True)
        states.append((h, c)); srcs[f'h{l}_τ-1'] = h; srcs[f'c{l}_τ-1'] = c
    y, _ = m.step_folded(u, states)
    y[0, :, ctr, ctr].sum().backward()
    ii, jj = torch.meshgrid(torch.arange(H), torch.arange(H), indexing='ij')
    cheb = torch.maximum((ii - ctr).abs(), (jj - ctr).abs())
    return {n: int(cheb[t.grad[0].abs().sum(0) > 0].max()) * p for n, t in srcs.items()}

ARCH_A = dict(patch=4, hidden=(64, 64), kernels=(3, 3), dilations=(1, 1))
_mA = ConvLSTMForecaster(**ARCH_A)
print('model A trainable parameters:', n_params(_mA))
assert n_params(_mA) == 480_784
REACH_A = measure_reach(ARCH_A)
print('model A empirical one-step reach per source (px, per axis):', REACH_A, '-> max', max(REACH_A.values()), 'px')

### Training and evaluation utilities
- Adam (lr 1e-3), batch 32, 30 epochs, final-epoch weights, one random window per training video per epoch.
- The loss is MSE averaged over pixels and *available* target frames per video, then over the batch.
- Seed `s` sets the initialisation (`torch.manual_seed`) and a dedicated `torch.Generator` that drives shuffling, window, speed and symmetry sampling.
- Epoch time covers the training pass only. The validation monitor runs outside the timer.

In [ ]:
def masked_mse(P, Y, mask):
    per = ((P - Y) ** 2).mean(dim=(2, 3, 4))                # (B, 5)
    return ((per * mask).sum(1) / mask.sum(1)).mean()

def apply_g_per_sample(X, Y, gi):
    X2, Y2 = X.clone(), Y.clone()
    for j, g in enumerate(G):
        sel = gi == j
        if sel.any():
            X2[sel] = apply_g(X[sel], g); Y2[sel] = apply_g(Y[sel], g)
    return X2, Y2

def sample_windows(vids, gen, speed_aug, sym_aug):
    B = vids.shape[0]
    if speed_aug:
        k = torch.randint(1, 4, (B,), generator=gen)                       # k ~ U{1,2,3}
        t = 1 + (torch.rand(B, generator=gen) * (20 - 6 * k)).long()       # t ~ U(allowed starts for k)
    else:
        k = torch.ones(B, dtype=torch.long)
        t = torch.randint(1, 12, (B,), generator=gen)                      # t ~ U{1..11}
    X, Y, mask, _ = make_windows(vids, t.to(DEVICE), k.to(DEVICE))
    if sym_aug:
        gi = torch.randint(0, 8, (B,), generator=gen).to(DEVICE)           # g ~ U(G), same g for X and Y
        X, Y = apply_g_per_sample(X, Y, gi)
    return X, Y, mask

@torch.no_grad()
def predict(f, X, bs=100):
    if isinstance(f, nn.Module): f.eval()
    return torch.cat([f(X[i:i + bs]) for i in range(0, len(X), bs)])

def vmse(P, Y):                              # per-video, per-horizon MSE: (N, 5)
    return ((P - Y) ** 2).mean(dim=(2, 3, 4))

@torch.no_grad()
def val_scores(f):
    out = {}
    for k in (1, 2, 3):
        X, Y, mask, _ = make_windows(val_u8, 1, k)
        e = vmse(predict(f, X), Y)
        out[f'E2_{k}x'] = e[:, :2].mean().item()
        if k == 1: out['E5_1x'] = e.mean().item()
    return out

def load_or_train(tag, arch, speed_aug, sym_aug, seed, epochs=None, monitor=True):
    epochs = epochs or EPOCHS
    path = f'{OUT}/ckpt/{tag}_s{seed}.pt'
    if os.path.exists(path):
        ck = torch.load(path, map_location=DEVICE)
        assert ck['arch'] == arch, f'{path} has a different architecture'
        m = ConvLSTMForecaster(**arch).to(DEVICE); m.load_state_dict(ck['state']); m.eval()
        print(f'loaded {path}'); return m, ck['log']
    random.seed(seed); np.random.seed(seed); torch.manual_seed(seed)
    if DEVICE.type == 'cuda': torch.cuda.manual_seed_all(seed)
    m = ConvLSTMForecaster(**arch).to(DEVICE)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    gen = torch.Generator().manual_seed(seed)
    log = {'loss': [], 'epoch_time': [], 'val': [], 'hardware': HARDWARE}
    for ep in range(epochs):
        m.train(); sync(); t0 = time.perf_counter()
        perm = torch.randperm(len(train_u8), generator=gen)
        tot, nb = torch.zeros((), device=DEVICE), 0
        for b in range(0, len(perm), BATCH):
            if MAX_BATCHES and nb >= MAX_BATCHES: break
            X, Y, mask = sample_windows(train_u8[perm[b:b + BATCH].to(DEVICE)], gen, speed_aug, sym_aug)
            loss = masked_mse(m(X), Y, mask)
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
            tot += loss.detach(); nb += 1
        sync(); log['epoch_time'].append(time.perf_counter() - t0); log['loss'].append((tot / nb).item())
        msg = f'[{tag} s{seed}] epoch {ep + 1:2d}/{epochs} loss {log["loss"][-1]:.5f} ({log["epoch_time"][-1]:.1f}s)'
        if monitor:
            log['val'].append(val_scores(m))
            msg += ' | val ' + ' '.join(f'{k}={fmt(v)}' for k, v in log['val'][-1].items())
        print(msg, flush=True)
    torch.save({'arch': arch, 'state': m.state_dict(), 'log': log, 'speed_aug': speed_aug, 'sym_aug': sym_aug}, path)
    m.eval(); return m, log

def blank(X):     return torch.zeros_like(X)
def copy_last(X): return X[:, -1:].expand_as(X).contiguous()

class Symmetrised(nn.Module):
    """f̄(X) = 1/8 Σ_g g⁻¹(f(g(X)))"""
    def __init__(self, f):
        super().__init__(); self.f = f
    def forward(self, X):
        return sum(apply_g(self.f(apply_g(X, g)), inv_g(g)) for g in G) / len(G)

@torch.no_grad()
def evaluate(f, cache=None):
    """Per-video test metrics. 1x/t=1 MSE per horizon, C/E under M and R, E(2) at 1-3x, E(5) at 1-2x."""
    if cache and os.path.exists(cache): return dict(np.load(cache))
    X, Y, _, _ = make_windows(test_u8, 1, 1)
    P = predict(f, X)
    r = {'mse_h': vmse(P, Y)}
    for nm, T in (('M', M), ('R', R)):
        PT = predict(f, T(X))
        r['C' + nm] = vmse(PT, T(P)).mean(1)
        r['E' + nm] = vmse(PT, T(Y)).mean(1)
    for k in (1, 2, 3):
        Xk, Yk, mk, _ = make_windows(test_u8, 1, k)
        e = vmse(predict(f, Xk), Yk)
        assert bool(mk[:, :2].all()) and (k == 3 or bool(mk.all()))
        r[f'E2_{k}x'] = e[:, :2].mean(1)
        if k <= 2: r[f'E5_{k}x'] = e.mean(1)
    r = {k: v.cpu().numpy() for k, v in r.items()}
    if cache: np.savez(cache, **r)
    return r

@torch.no_grad()
def time_inference(f):
    """Wall-clock to predict all original (1x, t=1) test videos at batch 32, timed after each batch is on the device."""
    if isinstance(f, nn.Module): f.eval()
    X, _, _, _ = make_windows(test_u8, 1, 1)
    batches = [X[i:i + BATCH].contiguous() for i in range(0, len(X), BATCH)]
    f(batches[0]); sync()                                     # warm-up (cuDNN autotune)
    total = 0.0
    for xb in batches:
        sync(); t0 = time.perf_counter(); f(xb); sync(); total += time.perf_counter() - t0
    return total

def scal(r, key):                          # dataset-level scalar from per-video arrays
    if key == 'E5': return float(r['mse_h'].mean())
    if key.startswith('MSE'): return float(r['mse_h'][:, int(key[3:]) - 1].mean())
    return float(r[key].mean())

MODELS, LOGS, RES, TIMES, PARAMS = {}, {}, {}, {}, {}

def run_model(name, tag, arch, speed_aug, sym_aug):
    """Train/load both seeds, evaluate on test, time inference."""
    RES[name], TIMES[name] = [], {'train': [], 'infer': []}
    for s in SEEDS:
        m, log = load_or_train(tag, arch, speed_aug, sym_aug, s)
        MODELS[(name, s)], LOGS[(name, s)] = m, log
        RES[name].append(evaluate(m, f'{OUT}/results/{tag}_s{s}.npz'))
        TIMES[name]['train'].append(float(np.mean(log['epoch_time'])))
        TIMES[name]['infer'].append(time_inference(m))
    PARAMS[name] = n_params(MODELS[(name, SEEDS[0])])

def run_baseline(name, f):
    RES[name] = [evaluate(f)]; TIMES[name] = {'train': [], 'infer': [time_inference(f)]}; PARAMS[name] = 0

### Train model A (seeds 0 and 1)

In [ ]:
run_model('A', 'A', ARCH_A, speed_aug=False, sym_aug=False)
run_baseline('Blank', blank)
run_baseline('Copy-last', copy_last)

fig, ax = plt.subplots(1, 2, figsize=(11, 3.5))
for s in SEEDS:
    ax[0].plot(range(1, EPOCHS + 1), LOGS[('A', s)]['loss'], label=f'seed {s}')
    ax[1].plot(range(1, EPOCHS + 1), [v['E5_1x'] for v in LOGS[('A', s)]['val']], label=f'seed {s}')
ax[0].set_title('model A train loss'); ax[1].set_title('model A val E(5) at 1x')
for a in ax: a.set_xlabel('epoch'); a.set_yscale('log'); a.legend()
fig.tight_layout(); fig.savefig(f'{OUT}/figs/modelA_training.png', dpi=120); plt.show()
print(f'hardware: {HARDWARE} | model A mean epoch time: {cell(TIMES["A"]["train"], ftime)} s')

### Table 2

In [ ]:
rows = []
for nm in ('A', 'Blank', 'Copy-last'):
    rows.append([nm] + [cell([scal(r, f'MSE{h}') for r in RES[nm]]) for h in range(1, 6)]
                + [cell([scal(r, 'E5') for r in RES[nm]]), f'{PARAMS[nm]:,}'])
table2 = pd.DataFrame(rows, columns=['Model', 'MSE1', 'MSE2', 'MSE3', 'MSE4', 'MSE5', 'E(5)', 'Params'])
save_table(table2, 'Table_2')

### Figure helpers, overlap detection and Figure 2
"Overlap" is detected from the frames: the bright area (>0.2) falls well below the video's median area and/or the digits merge into fewer connected components.
You can override the automatic picks with `FIG2_VIDEOS` (test-split indices 0–999, i.e. dataset index − 9000).

In [ ]:
def img(t): return t.detach().float().cpu().reshape(64, 64).numpy()

def grid_figure(rows, row_labels=None, titles=None, suptitle=None, fname=None, sep_col=None, scale=1.15, show=True):
    nr, nc = len(rows), max(len(r) for r in rows)
    fig, axes = plt.subplots(nr, nc, figsize=(nc * scale + (1.6 if row_labels else 0), nr * scale + 0.6), squeeze=False)
    for i in range(nr):
        for j in range(nc):
            a = axes[i, j]; a.set_xticks([]); a.set_yticks([])
            if j < len(rows[i]) and rows[i][j] is not None:
                a.imshow(rows[i][j], cmap='gray', vmin=0, vmax=1)
            else:
                a.axis('off')
            if titles is not None and titles[i] is not None and j < len(titles[i]) and titles[i][j]:
                a.set_title(titles[i][j], fontsize=7)
            if sep_col is not None and j == sep_col:
                for sp in a.spines.values(): sp.set_edgecolor('tab:red'); sp.set_linewidth(1.5)
        if row_labels: axes[i, 0].set_ylabel(row_labels[i], rotation=0, ha='right', va='center', fontsize=8)
    if suptitle: fig.suptitle(suptitle, fontsize=9)
    fig.tight_layout()
    if fname: fig.savefig(fname, dpi=150)
    if show: plt.show()
    else: plt.close(fig)
    return fig

_bin = (test_u8[:, :, 0] > 51).cpu().numpy()                  # intensity > 0.2
AREA = _bin.sum((2, 3)).astype(float)                         # (N, 20)
NCOMP = np.array([[cv2.connectedComponents(f.astype(np.uint8))[0] - 1 for f in v] for v in _bin])
AREA_RATIO = AREA / np.median(AREA, axis=1, keepdims=True)

def overlaps(i, frames, thr=0.93):                            # frames are 1-based
    idx = np.array(frames) - 1
    return bool((AREA_RATIO[i, idx] < thr).any() or (NCOMP[i, idx] < 2).any())

FIG2_VIDEOS = None                                             # e.g. [3, 17, 42]
if FIG2_VIDEOS is None:
    clean = [i for i in range(N_TEST) if not overlaps(i, range(1, 11))]
    ov = int(np.argmin(AREA_RATIO[:, 5:10].min(1)))          # strongest overlap within the target frames 6-10
    FIG2_VIDEOS = clean[:2] + [ov]
print('Figure 2 videos (test idx):', FIG2_VIDEOS, '-> dataset idx', [9000 + i for i in FIG2_VIDEOS])

def fig2_like(f, vids, fname, label):
    X, Y, _, _ = make_windows(test_u8[vids], 1, 1); P = predict(f, X)
    rows, labels, titles = [], [], []
    for n, i in enumerate(vids):
        rows += [[img(X[n, h]) for h in range(5)], [img(Y[n, h]) for h in range(5)], [img(P[n, h]) for h in range(5)]]
        labels += [f'video {i}\ninput', 'ground truth', f'{label}\nprediction']
        titles += [[f'frame {h + 1}' for h in range(5)], [f'frame {h + 6}' for h in range(5)], None]
    return grid_figure(rows, labels, titles, fname=fname,
                       suptitle=f'{label}: input (frames 1-5), ground truth and prediction (frames 6-10); '
                                f'last video has overlapping digits')

fig2_like(MODELS[('A', 0)], FIG2_VIDEOS, f'{OUT}/figs/fig2_modelA.png', 'Model A (seed 0)');

---
## ⛔ STOP: Q2.5
Answer **Q2.3** (reach) and **Q2.4** in your report. Then write your **Q2.5 predictions** (mirror test, rotation test, breaking speed), **commit and push them to GitHub**, and only then set `Q2_5_COMMITTED = True` in the config cell.

*(Q2.5 must appear in the report exactly as first written, even if it turns out wrong.)*

---
# 3. Testing the Baseline
### Table 3: consistency gap and transformed error

In [ ]:
%%stage 2
rows = []
for nm in ('A', 'Blank', 'Copy-last'):
    rows.append([nm] + [cell([scal(r, k) for r in RES[nm]]) for k in ('E5', 'CM', 'EM', 'CR', 'ER')])
table3 = pd.DataFrame(rows, columns=['Model', 'E(5)', 'C_M', 'E_M', 'C_R', 'E_R'])
save_table(table3, 'Table_3')

### Figure 3: the largest per-video C_M and C_R for model A (seed 0), with GIFs
Rows: (1) X | f(X), (2) T(X) | T(Y), (3) T(X) | f(T(X)), (4) T(X) | T(f(X)). Each GIF steps through horizons h = 1…5, keeping the four rows: the left column is the row's last input frame and the right column is horizon h.

In [ ]:
%%stage 2
TRANSFORMS = {'M': M, 'R': R}

def panel(f, i, Tn, label, fname_png, fname_gif):
    T = TRANSFORMS[Tn]
    x, y = to_f(test_u8[i:i + 1, :5]), to_f(test_u8[i:i + 1, 5:10])
    fx, ftx = predict(f, x), predict(f, T(x))
    pairs = [(x, fx), (T(x), T(y)), (T(x), ftx), (T(x), T(fx))]
    names = [f'(1) X | f(X)', f'(2) {Tn}(X) | {Tn}(Y)', f'(3) {Tn}(X) | f({Tn}(X))', f'(4) {Tn}(X) | {Tn}(f(X))']
    rows = [[img(a[0, h]) for h in range(5)] + [img(b[0, h]) for h in range(5)] for a, b in pairs]
    titles = [[f'in {h + 1}' for h in range(5)] + [f'h={h + 1}' for h in range(5)]] + [None] * 3
    grid_figure(rows, names, titles, fname=fname_png, sep_col=5,
                suptitle=f'{label}, T={Tn}, test video {i} (dataset {9000 + i})')
    frames = []
    for h in range(5):
        fig = grid_figure([[img(a[0, 4]), img(b[0, h])] for a, b in pairs], names,
                          [['last input', f'h = {h + 1} (frame {h + 6})']] + [None] * 3,
                          suptitle=f'{label}, T={Tn}, video {i}', scale=1.6, show=False)
        fig.canvas.draw(); frames.append(Image.fromarray(np.asarray(fig.canvas.buffer_rgba())[..., :3].copy()))
        plt.close(fig)
    frames[0].save(fname_gif, save_all=True, append_images=frames[1:], duration=700, loop=0)
    return fname_gif

rA0 = RES['A'][SEEDS.index(0)]
FIG3_PICKS = ([(int(i), 'M') for i in np.argsort(-rA0['CM'])[:2]] +
              [(int(i), 'R') for i in np.argsort(-rA0['CR'])[:2]])
print('Figure 3 picks (test idx, T):', FIG3_PICKS)
for n, (i, Tn) in enumerate(FIG3_PICKS):
    print(f'video {i}: C_{Tn} = {fmt(rA0["C" + Tn][i])}, E_{Tn} = {fmt(rA0["E" + Tn][i])}, E(5) = {fmt(rA0["mse_h"][i].mean())}')
    g = panel(MODELS[('A', 0)], i, Tn, 'Model A (seed 0)',
              f'{OUT}/figs/fig3_{chr(97 + n)}_A_{Tn}_v{i}.png', f'{OUT}/gifs/fig3_{chr(97 + n)}_A_{Tn}_v{i}.gif')
    print('GIF ->', g.replace(OUT + '/', ''))

### Figure 4: per-video C_M and C_R histograms (model A, seed 0)

In [ ]:
%%stage 2
fig, ax = plt.subplots(1, 2, figsize=(12, 3.8))
for a, Tn in zip(ax, ('M', 'R')):
    v = rA0['C' + Tn]; v = v[v > 0]
    a.hist(v, bins=np.logspace(np.log10(v.min()), np.log10(v.max()), 40), edgecolor='k')
    a.set_xscale('log'); a.set_xlabel(f'per-video C_{Tn}'); a.set_ylabel('# test videos')
    for n, (i, T2) in enumerate(FIG3_PICKS):
        if T2 == Tn:
            a.axvline(rA0['C' + Tn][i], color='r', ls='--'); a.text(rA0['C' + Tn][i], a.get_ylim()[1] * 0.9, f' fig3{chr(97 + n)}\n v{i}', color='r', fontsize=8)
    a.set_title(f'C_{Tn}: mean {fmt(v.mean())}, median {fmt(np.median(v))}')
fig.suptitle('Figure 4: per-video consistency gaps of model A (seed 0) on the test set; red = videos in Figure 3')
fig.tight_layout(); fig.savefig(f'{OUT}/figs/fig4_hist_C.png', dpi=150); plt.show()

### Faster motion: Table 4 and Figure 5
For Figure 5, the selected test video has no digit overlap in any displayed **true** frame (1×: 6,7; 2×: 11,13; 3×: 16,19). Override with `FIG5_VIDEO` if you like.

In [ ]:
%%stage 2
rows = []
for nm in ('A', 'Copy-last'):
    rows.append([nm] + [cell([scal(r, k) for r in RES[nm]]) for k in ('E2_1x', 'E2_2x', 'E2_3x', 'E5_1x', 'E5_2x')])
table4 = pd.DataFrame(rows, columns=['Model', 'E(2) 1x', 'E(2) 2x', 'E(2) 3x', 'E(5) 1x', 'E(5) 2x'])
save_table(table4, 'Table_4')

FIG5_VIDEO = None
if FIG5_VIDEO is None:
    FIG5_VIDEO = next(i for i in range(N_TEST) if not overlaps(i, [6, 7, 11, 13, 16, 19], thr=0.95))
print('Figure 5 video (test idx):', FIG5_VIDEO)

def fig5_like(f, i, label, fname):
    rows, titles, labels = [], [], []
    for k in (1, 2, 3):
        X, Y, mask, fr = make_windows(test_u8[i:i + 1], 1, k); P = predict(f, X); fr = fr[0].tolist()
        rows.append([img(X[0, h]) for h in range(5)] + [img(Y[0, h]) for h in range(2)] + [img(P[0, h]) for h in range(2)])
        titles.append([f'in fr {fr[h]}' for h in range(5)] + [f'true fr {fr[5 + h]}' for h in range(2)]
                      + [f'pred h={h + 1}' for h in range(2)])
        labels.append(f'{k}x')
    return grid_figure(rows, labels, titles, fname=fname, sep_col=7,
                       suptitle=f'{label}: test video {i} (dataset {9000 + i}); inputs, first two true and predicted frames')

fig5_like(MODELS[('A', 0)], FIG5_VIDEO, 'Model A (seed 0)', f'{OUT}/figs/fig5_A_speed.png');

---
## ⛔ STOP: Q2.11
Answer **Q2.6–Q2.10** in your report. Then write your **Q2.11 prediction** (at most three sentences: which of B/C/D fixes which failure), **commit and push it**, and only then set `Q2_11_COMMITTED = True`.

---
# 4. What Can Data Fix?
### Model D candidates
All candidates use the same rollout and training protocol as model C (speed augmentation). Each must have ≤ 721,176 parameters and a strictly larger reach than A.
- `D_k5`: 5×5 kernels, 44 hidden channels, patch 4 (dense receptive field)
- `D_p8`: 8×8 patches (8×8 grid), 64 hidden channels, 3×3 kernels
- `D_dil2`: same as A but with dilation 2 in both layers (same parameter count as A; leaves gaps in the receptive field)

The selection uses the **validation split only**: a short seed-0 run per candidate, choosing the lowest mean val E(2) over 1×/2×/3×.

In [ ]:
%%stage 3
D_CANDIDATES = {
    'D_k5':   dict(patch=4, hidden=(44, 44), kernels=(5, 5), dilations=(1, 1)),
    'D_p8':   dict(patch=8, hidden=(64, 64), kernels=(3, 3), dilations=(1, 1)),
    'D_dil2': dict(patch=4, hidden=(64, 64), kernels=(3, 3), dilations=(2, 2)),
}
rows = []
for nm, a in D_CANDIDATES.items():
    npar, rch = n_params(ConvLSTMForecaster(**a)), measure_reach(a)
    assert npar <= 721_176 and max(rch.values()) > max(REACH_A.values()), nm
    rows.append([nm, f'{npar:,}', max(rch.values()), str(rch)])
rows.append(['A (ref)', f'{n_params(_mA):,}', max(REACH_A.values()), str(REACH_A)])
display(pd.DataFrame(rows, columns=['candidate', 'params', 'reach px (max)', 'reach per source']))

if RUN_D_SELECTION:
    sel = []
    for nm, a in D_CANDIDATES.items():
        m, log = load_or_train(f'sel_{nm}', a, True, False, seed=0, epochs=SEL_EPOCHS, monitor=False)
        v = val_scores(m)
        sel.append(dict(candidate=nm, **{k: fmt(x) for k, x in v.items()},
                        mean_E2=np.mean([v['E2_1x'], v['E2_2x'], v['E2_3x']]), epoch_s=fmt(np.mean(log['epoch_time']))))
        del m
    sel_df = pd.DataFrame(sel).sort_values('mean_E2')
    save_table(sel_df.assign(mean_E2=sel_df.mean_E2.map(fmt)), 'D_selection_validation')
    D_CHOICE = sel_df.iloc[0]['candidate']
else:
    D_CHOICE = D_DEFAULT
ARCH_D = D_CANDIDATES[D_CHOICE]
print('Model D =', D_CHOICE, ARCH_D)

### Train B, C, D (and optionally D+B): 2 seeds each, 30 epochs

In [ ]:
%%stage 3
SPECS = {'B': ('B', ARCH_A, False, True),
         'C': ('C', ARCH_A, True, False),
         'D': (f'D_{D_CHOICE}', ARCH_D, True, False)}
if TRAIN_DB or FINAL == 'D+B':
    SPECS['D+B'] = (f'DB_{D_CHOICE}', ARCH_D, True, True)
for nm, (tag, arch, sp, sy) in SPECS.items():
    run_model(nm, tag, arch, sp, sy)

### Table 5

In [ ]:
%%stage 3
T5_COLS = ['E5', 'CM', 'EM', 'CR', 'ER', 'E2_1x', 'E2_2x', 'E2_3x']
T5_HEAD = ['E(5) 1x', 'C_M', 'E_M', 'C_R', 'E_R', 'E(2) 1x', 'E(2) 2x', 'E(2) 3x']

def results_row(nm, res_name=None, time_name=None, csci=False):
    res_name = res_name or nm; time_name = time_name or nm
    vals = []
    for k in T5_COLS:
        f = fmt_sci if (csci and k in ('CM', 'CR')) else fmt
        vals.append(cell([scal(r, k) for r in RES[res_name]], f))
    return ([nm] + vals + [f'{PARAMS[res_name]:,}', cell(TIMES[time_name]['train'], ftime), cell(TIMES[res_name]['infer'], ftime)])

T5_NAMES = ['A', 'B', 'C', 'D'] + (['D+B'] if 'D+B' in RES else [])
table5 = pd.DataFrame([results_row(n) for n in T5_NAMES],
                      columns=['Model'] + T5_HEAD + ['Params', 'Train s/epoch', 'Inference s'])
save_table(table5, 'Table_5')
print(f'hardware: {HARDWARE}; model D = {D_CHOICE} {ARCH_D}; reach D = {max(measure_reach(ARCH_D).values())} px '
      f'vs A = {max(REACH_A.values())} px')

---
# 5. The Final Model
Symmetrised forecaster f̄(X) = ⅛ Σ_g g⁻¹(f(g(X))). No retraining is needed. C_M and C_R should be 0 up to floating-point error.
Inference time includes all 8 transforms, forward passes, inverse transforms and the averaging.

In [ ]:
%%stage 3
SYM = {'B': 'Sym(B)', 'D': 'Sym(D)'}
if FINAL == 'D+B': SYM['D+B'] = 'Sym(D+B)'
for base, nm in SYM.items():
    RES[nm], TIMES[nm] = [], {'train': TIMES[base]['train'], 'infer': []}
    for s in SEEDS:
        f = Symmetrised(MODELS[(base, s)]).eval(); MODELS[(nm, s)] = f
        RES[nm].append(evaluate(f, f'{OUT}/results/{SPECS[base][0]}_sym_s{s}.npz'))
        TIMES[nm]['infer'].append(time_inference(f))
    PARAMS[nm] = PARAMS[base]
FINAL_NAME = SYM[FINAL]

table6 = pd.DataFrame([results_row(nm, csci=True) for nm in SYM.values()],
                      columns=['Model'] + T5_HEAD + ['Params', 'Train s/epoch (underlying)', 'Inference s'])
save_table(table6, 'Table_6')
print('final model:', FINAL_NAME, '| max per-video C_M, C_R of final (seed 0):',
      fmt_sci(RES[FINAL_NAME][0]['CM'].max()), fmt_sci(RES[FINAL_NAME][0]['CR'].max()))

### Figure 6: (a) Figure 3 and (b) Figure 5 repeated with the final model (seed 0) on the same videos

In [ ]:
%%stage 3
fin0 = MODELS[(FINAL_NAME, 0)]
rF0 = RES[FINAL_NAME][SEEDS.index(0)]
for n, (i, Tn) in enumerate(FIG3_PICKS):
    print(f'video {i}: model A  C_{Tn}={fmt(rA0["C" + Tn][i])} E_{Tn}={fmt(rA0["E" + Tn][i])} | '
          f'{FINAL_NAME}  C_{Tn}={fmt(rF0["C" + Tn][i])} E_{Tn}={fmt(rF0["E" + Tn][i])}')
    g = panel(fin0, i, Tn, f'{FINAL_NAME} (seed 0)',
              f'{OUT}/figs/fig6a_{chr(97 + n)}_final_{Tn}_v{i}.png', f'{OUT}/gifs/fig6a_{chr(97 + n)}_final_{Tn}_v{i}.gif')
    print('GIF ->', g.replace(OUT + '/', ''))
fig5_like(fin0, FIG5_VIDEO, f'{FINAL_NAME} (seed 0)', f'{OUT}/figs/fig6b_final_speed.png');

### Package outputs
Download `pa2_q2.zip` from the Output tab. Put `gifs/` (and ideally the figures and tables) in your GitHub repo and cite the GIF paths in the Figure 3 and 6 captions.

In [ ]:
info = dict(hardware=HARDWARE, torch=torch.__version__, epochs=EPOCHS, n_test=N_TEST, quick=QUICK,
            fig2_videos=FIG2_VIDEOS, stage2=stage_unlocked(2), stage3=stage_unlocked(3))
for k in ('FIG3_PICKS', 'FIG5_VIDEO', 'D_CHOICE', 'FINAL_NAME'):
    if k in globals(): info[k] = globals()[k]
with open(f'{OUT}/run_info.json', 'w') as fh: json.dump(info, fh, indent=2, default=str)
print(json.dumps(info, indent=2, default=str))
zip_path = shutil.make_archive(OUT, 'zip', os.path.dirname(OUT), os.path.basename(OUT))
print('wrote', zip_path, f'({os.path.getsize(zip_path) / 1e6:.1f} MB)')